# Demo 1：Partition Discovery

任务：`partition-discovery`。本 Demo 把相组成、溶剂和萃取条件视为对隐藏分配构成律的候选干预。重点不是操作仪器，而是观察哪些公开反馈能够支持或否定一个局部分配 world model。

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'chemworld').exists()), None)
if ROOT is None:
    raise RuntimeError('请从 ChemWorld 仓库内启动 notebook')
for path in (ROOT, ROOT / 'src'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
du = importlib.import_module('notebooks.task_demos.demo_utils')

TASK_ID = 'partition-discovery'
SEED = 0

## 1. 公开任务合同

先确认 Agent 能看到的目标、预算、操作和指标。以下步骤不读取 hidden state、分配系数或机制标签。

In [2]:
card = du.task_card(TASK_ID)
pd.DataFrame({
    'field': ['task_id', 'motivation', 'budget', 'episode_mode', 'success_metrics'],
    'value': [card['task_id'], card['scientific_motivation'], card['budget'], card['episode_mode'], card['reward_leaderboard_metric']['success_metrics']],
})

,field,value
0,task_id,partition-discovery
1,motivation,Learn unknown solvent/product partition behavi...
2,budget,48
3,episode_mode,campaign
4,success_metrics,"[phase_ratio, product_in_organic, product_in_a..."


## 2. 构造候选干预

归一化向量会通过公开 recipe adapter 映射成不同的溶剂、相体积、混合与静置条件。`low/mid/high` 只是三组确定性探针，不代表最优策略。

In [3]:
vectors = du.standard_probe_vectors(TASK_ID)
mid_recipe = du.recipe_frame(TASK_ID, vectors['mid'])
mid_recipe

,step,operation,payload
0,1,add_solvent,"{'operation': 'add_solvent', 'volume_L': 0.02,..."
1,2,add_phase,"{'operation': 'add_phase', 'phase': 'aqueous',..."
2,3,add_extractant,"{'operation': 'add_extractant', 'extractant': ..."
3,4,mix,"{'operation': 'mix', 'duration_s': 330.0, 'sti..."
4,5,settle,"{'operation': 'settle', 'duration_s': 660.0}"
5,6,measure,"{'operation': 'measure', 'instrument': 'hplc'}"
6,7,separate_phase,"{'operation': 'separate_phase', 'target_phase'..."
7,8,measure,"{'operation': 'measure', 'instrument': 'hplc'}"
8,9,terminate,{'operation': 'terminate'}
9,10,measure,"{'operation': 'measure', 'instrument': 'final_..."


## 3. 执行并读取反馈

比较三组候选干预的公开 `processed_estimate` 和 final-assay `leaderboard_score`。这里应关注 `product_in_organic`、`product_in_aqueous` 与 `phase_ratio` 的联合变化。

In [4]:
comparison = du.compare_vectors(TASK_ID, vectors, seed=SEED)
columns = ['candidate', 'leaderboard_score', 'phase_ratio', 'product_in_organic', 'product_in_aqueous', 'cost', 'all_actions_valid']
comparison[[column for column in columns if column in comparison]]

,candidate,leaderboard_score,phase_ratio,product_in_organic,product_in_aqueous,cost,all_actions_valid
0,low,0.515215,0.555654,0.633814,0.009035,0.0,True
1,mid,0.583671,0.550841,0.759266,0.009035,0.0,True
2,high,0.576773,0.548539,0.749317,0.009035,0.0,True


### 查看一次完整运行中的测量反馈

同一实验中的不同测量是更新 belief 的证据，而不是隐藏真值。

In [5]:
run = du.run_vector(TASK_ID, vectors['mid'], seed=SEED)
measurement_feedback = du.measurement_trace(run)
measurement_feedback

,step,operation,reward,leaderboard_score,observed_keys,processed_estimate
0,6,measure,0.129747,NaN,"yield, selectivity, byproduct_signal, cost, sa...","{'yield': 0.7572598244603723, 'selectivity': 0..."
1,8,measure,-0.007115,NaN,"yield, selectivity, byproduct_signal, cost, sa...","{'yield': 0.7473062389775678, 'selectivity': 0..."
2,10,measure,0.461040,0.583671,"yield, selectivity, conversion, byproduct_sign...","{'yield': 0.7513574576790308, 'selectivity': 0..."


## 4. 同一干预，不同隐藏规律

教师端用 `constitutive_law_family` 构造两个 opaque worlds，再执行完全相同的公开干预。Agent 在正式评测中不会看到这一名称，只能根据反馈判断旧 world model 是否仍然适用。

In [6]:
paired_worlds = du.compare_hidden_worlds(
    TASK_ID, vectors['mid'], mechanism_mode='constitutive_law_family', seed=SEED
)
paired_worlds

,opaque_world,phase_ratio,product_in_organic,product_in_aqueous,leaderboard_score,cost,safety_risk,all_actions_valid
0,World A,0.550841,0.759266,0.009035,0.583671,0.0,None,True
1,World B,0.550841,0.778920,0.009035,0.594956,0.0,None,True


## 5. 留给 Agent 的能力问题

环境只提供交互和评测。Agent 可以使用固定权重加上下文、显式 belief、外部记忆或自己的在线学习方法。

In [7]:
capability_probe = {
    'current_evidence': comparison.to_dict(orient='records'),
    'prediction_query': '预测一个未执行相体积条件下 organic/aqueous 分配及不确定性。',
    'next_intervention_query': '选择最能区分固定分配系数与条件依赖构成律的下一次实验。',
    'evaluation_note': '比较预测是否随新反馈改善，而不是给自然语言解释打分。',
}
capability_probe

{'current_evidence': [{'candidate': 'low',
   'phase_ratio': 0.5556540275545242,
   'product_in_organic': 0.6338144463752431,
   'product_in_aqueous': 0.009034701816518087,
   'leaderboard_score': 0.5152146046131233,
   'cost': 0.0,
   'safety_risk': None,
   'all_actions_valid': True,
   'operation_count': 10},
  {'candidate': 'mid',
   'phase_ratio': 0.5508411933299252,
   'product_in_organic': 0.7592661625081818,
   'product_in_aqueous': 0.009034701816518087,
   'leaderboard_score': 0.5836711285190952,
   'cost': 0.0,
   'safety_risk': None,
   'all_actions_valid': True,
   'operation_count': 10},
  {'candidate': 'high',
   'phase_ratio': 0.5485394030485954,
   'product_in_organic': 0.7493168271417153,
   'product_in_aqueous': 0.009034701816518087,
   'leaderboard_score': 0.5767731796654314,
   'cost': 0.0,
   'safety_risk': None,
   'all_actions_valid': True,
   'operation_count': 10}],
 'prediction_query': '预测一个未执行相体积条件下 organic/aqueous 分配及不确定性。',
 'next_intervention_query': '选择最能